In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 9.7 MB/s eta 0:00:00


In [2]:
# Import necessary libraries

import optuna

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source.

import pandas as pd

# Load the Pima Indian Diabetes Dataset (from UCI repository)

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = [
    'Pregnancies',
    'Glucose',
    'BloodPressure',
    'SkinThickness',
    'Insulin',
    'BMI',
    'DiabetesPedigreeFunction',
    'Age',
    'Outcome'
]

# Load the dataset

df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = [
    'Glucose',
    'BloodPressure',
    'SkinThickness',
    'Insulin',
    'BMI'
]

df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# Split into features (X) and target (y)

X = df.drop('Outcome', axis=1)
y = df['Outcome']


# Split data into training and test sets (70% train, 30% test)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)


# Optional: Scale the data for better model performance

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# Check the shape of the data

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: (537, 8)
Test set shape: (231, 8)


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):

    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=3,
        scoring='accuracy'
    ).mean()

    # Return the accuracy score for Optuna to maximize
    return score


# Create a study object and optimize the objective function
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler()
)

# Run 50 trials to find the best hyperparameters
study.optimize(objective, n_trials=50)

[I 2026-07-30 15:58:02,573] A new study created in memory with name: no-name-2817b951-b2b9-4f61-b27f-24a5e4d216ca
[I 2026-07-30 15:58:04,177] Trial 0 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 113, 'max_depth': 7}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-07-30 15:58:05,606] Trial 1 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 142, 'max_depth': 5}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-07-30 15:58:06,814] Trial 2 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 89, 'max_depth': 12}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-07-30 15:58:09,836] Trial 3 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 152, 'max_depth': 11}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-07-30 15:58:10,772] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 124, 'max_depth': 8}. Best is trial 0 with value: 0.77281191

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):

    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score

In [11]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())

study.optimize(objective, n_trials=60)

[I 2026-07-30 16:06:09,628] A new study created in memory with name: no-name-ae35ce89-2fef-4f3b-b1ae-66ba3f2dd6c5
[I 2026-07-30 16:06:12,081] Trial 0 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 66, 'max_depth': 15}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-07-30 16:06:16,101] Trial 1 finished with value: 0.7802607076350093 and parameters: {'n_estimators': 133, 'max_depth': 18}. Best is trial 1 with value: 0.7802607076350093.
[I 2026-07-30 16:06:21,536] Trial 2 finished with value: 0.7579143389199254 and parameters: {'n_estimators': 197, 'max_depth': 3}. Best is trial 1 with value: 0.7802607076350093.
[I 2026-07-30 16:06:22,683] Trial 3 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 62, 'max_depth': 18}. Best is trial 1 with value: 0.7802607076350093.
[I 2026-07-30 16:06:26,599] Trial 4 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 187, 'max_depth': 17}. Best is trial 1 with value: 0.7802607

In [13]:
# print the best result
print(f'Best Trial Accuracy: {study.best_trial.value}')
print(f'best hyper parameter: {study.best_trial.params}')

Best Trial Accuracy: 0.7802607076350094
best hyper parameter: {'n_estimators': 70, 'max_depth': 18}


In [14]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


In [15]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())

study.optimize(objective, n_trials=50)

[I 2026-07-30 16:15:44,365] A new study created in memory with name: no-name-62a38227-d48e-4d76-9b3a-7165ffe1346c
[I 2026-07-30 16:15:47,981] Trial 0 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 180, 'max_depth': 17}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-07-30 16:15:53,612] Trial 1 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 195, 'max_depth': 14}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-07-30 16:15:58,587] Trial 2 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 188, 'max_depth': 14}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-07-30 16:16:01,881] Trial 3 finished with value: 0.7765363128491619 and parameters: {'n_estimators': 185, 'max_depth': 16}. Best is trial 3 with value: 0.7765363128491619.
[I 2026-07-30 16:16:03,380] Trial 4 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 68, 'max_depth': 13}. Best is trial 3 with value: 0.77653

# Visualization

In [17]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [18]:
# 1. Optimization History
plot_optimization_history(study).show()

In [19]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [20]:
# slice plot

plot_slice(study).show()

In [21]:
# countour plot

plot_contour(study).show()

In [22]:
# plto param

plot_param_importances(study).show()

In [23]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [24]:
# Define the objective function for Optuna
def objective(trial):

    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical(
        'classifier',
        ['SVM', 'RandomForest', 'GradientBoosting']
    )

    if classifier_name == 'SVM':

        # SVM hyperparameters
        C = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical(
            'kernel',
            ['linear', 'rbf', 'poly', 'sigmoid']
        )
        gamma = trial.suggest_categorical(
            'gamma',
            ['scale', 'auto']
        )

        model = SVC(
            C=C,
            kernel=kernel,
            gamma=gamma,
            random_state=42
        )

    elif classifier_name == 'RandomForest':

        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':

        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=3,
        scoring='accuracy'
    ).mean()

    return score

In [25]:
study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=100)

[I 2026-07-30 16:42:08,282] A new study created in memory with name: no-name-d622b94b-ebdc-409e-af86-a30795d3559a
[I 2026-07-30 16:42:10,519] Trial 0 finished with value: 0.7653631284916201 and parameters: {'classifier': 'RandomForest', 'n_estimators': 184, 'max_depth': 13, 'min_samples_split': 3, 'min_samples_leaf': 1, 'bootstrap': False}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-07-30 16:42:13,599] Trial 1 finished with value: 0.7392923649906891 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 135, 'learning_rate': 0.10733551529460444, 'max_depth': 11, 'min_samples_split': 2, 'min_samples_leaf': 10}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-07-30 16:42:13,670] Trial 2 finished with value: 0.7746741154562384 and parameters: {'classifier': 'SVM', 'C': 0.9172313963086588, 'kernel': 'rbf', 'gamma': 'scale'}. Best is trial 2 with value: 0.7746741154562384.
[I 2026-07-30 16:42:17,990] Trial 3 finished with value: 0.7448789571694601 and parame

In [26]:
# Retrieve the best trial
best_trial = study.best_trial

print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.12797651035308985, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


In [27]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.765363,2026-07-30 16:42:08.292982,2026-07-30 16:42:10.519228,0 days 00:00:02.226246,NaN,False,RandomForest,NaN,NaN,NaN,13.0,1.0,3.0,184.0,COMPLETE
1,1,0.739292,2026-07-30 16:42:10.522829,2026-07-30 16:42:13.599791,0 days 00:00:03.076962,NaN,NaN,GradientBoosting,NaN,NaN,0.107336,11.0,10.0,2.0,135.0,COMPLETE
2,2,0.774674,2026-07-30 16:42:13.602549,2026-07-30 16:42:13.670091,0 days 00:00:00.067542,0.917231,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.744879,2026-07-30 16:42:13.672158,2026-07-30 16:42:17.990486,0 days 00:00:04.318328,NaN,NaN,GradientBoosting,NaN,NaN,0.036772,16.0,9.0,9.0,234.0,COMPLETE
4,4,0.765363,2026-07-30 16:42:17.991726,2026-07-30 16:42:19.107489,0 days 00:00:01.115763,NaN,NaN,GradientBoosting,NaN,NaN,0.083542,3.0,7.0,4.0,177.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.787709,2026-07-30 16:43:06.002086,2026-07-30 16:43:06.037816,0 days 00:00:00.035730,0.100512,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.715084,2026-07-30 16:43:06.039244,2026-07-30 16:43:06.074916,0 days 00:00:00.035672,0.157254,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.789572,2026-07-30 16:43:06.076258,2026-07-30 16:43:06.115232,0 days 00:00:00.038974,0.121819,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.785847,2026-07-30 16:43:06.116542,2026-07-30 16:43:06.157901,0 days 00:00:00.041359,0.231839,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


# ye chcek karna ke har algo kitni bar run hua

In [28]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,78
GradientBoosting,12
RandomForest,10


In [29]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.741310
RandomForest,0.766480
SVM,0.776918
